In [1]:
import pandas as pd
import re
import string
from itertools import combinations
from collections import Counter

import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

print("Imports complete.")


Imports complete.


In [2]:
# Load Data

df = pd.read_csv("comments_cleaned.csv")

keep_cols = [c for c in [
    "video_id", "video_title", "author",
    "comment_published_at", "comment_like_count", "clean_comment"
] if c in df.columns]

df = df[keep_cols].copy()
df = df.dropna(subset=["clean_comment"]).reset_index(drop=True)

print(f"Loaded {len(df):,} rows")
print(f"Columns: {list(df.columns)}")
df.head(3)


Loaded 6,241 rows
Columns: ['video_id', 'video_title', 'author', 'comment_published_at', 'comment_like_count', 'clean_comment']


,video_id,video_title,author,comment_published_at,comment_like_count,clean_comment
0,laZpTO7IFtA,Is 67 just brain rot?,@ExoticNibbles2,2025-10-14T22:24:09Z,7902,nothing makes me feel older then watching a 15...
1,laZpTO7IFtA,Is 67 just brain rot?,@andrewbunnell7576,2025-10-13T20:12:09Z,7692,6 x 7 = 42 the answer to life the universe and...
2,laZpTO7IFtA,Is 67 just brain rot?,@ColinPaddock,2025-10-14T06:39:54Z,6194,all i know is 6 is afraid of 7.


In [3]:
# Create network_text, Lowercase, Remove Punctuation and Extra Spaces

df["network_text"] = df["clean_comment"].astype(str)

# Lowercase (clean_comment already is, but enforce)
df["network_text"] = df["network_text"].str.lower()

# Remove punctuation except hyphens and underscores (needed for phrase joining later)
df["network_text"] = df["network_text"].apply(
    lambda t: re.sub(r"[^\w\s\-]", " ", t)
)

# Collapse extra whitespace
df["network_text"] = df["network_text"].apply(
    lambda t: re.sub(r"\s+", " ", t).strip()
)

print("Preprocessing step 3 complete.")
print("Sample:", df["network_text"].iloc[0])


Preprocessing step 3 complete.
Sample: nothing makes me feel older then watching a 15 minute video breaking down slang the young kids use these days


In [4]:
# Standardise Terms and Convert Multi-word Phrases

# Order matters: more specific patterns first.
# Each tuple is (regex_pattern, replacement).
TEXT_REPLACEMENTS = [
    # Brainrot variants
    (r"\bbrain[\s\-]?rot\b", "brainrot"),
    (r"\bbrain\s+rotting\b", "brainrot"),
    (r"\bbrain\s+dead\b", "braindead"),
    (r"\bbrain\s+cells?\b", "braincells"),
    (r"\bbrain\s+worms?\b", "brainworm"),
    (r"\bbrain\s+damage\b", "brain_damage"),

    # Platform phrases
    (r"\bsocial\s+media\b", "social_media"),
    (r"\byoutube\s+shorts\b", "youtube_shorts"),
    (r"\btik[\s\-]?tok\b", "tiktok"),
    (r"\bfor\s+you\s+page\b", "fyp"),
    (r"\bshort[\s\-]?form\b", "shortform"),
    (r"\blong[\s\-]?form\b", "longform"),
    (r"\bshort\s+videos?\b", "short_video"),
    (r"\blong\s+videos?\b", "long_video"),
    (r"\bvideo\s+essays?\b", "video_essay"),

    # Attention / behaviour phrases
    (r"\battention\s+spans?\b", "attention_span"),
    (r"\bscreen\s+time\b", "screen_time"),
    (r"\bdoom[\s\-]?scrolling\b", "doomscrolling"),
    (r"\bmindless\s+scrolling\b", "mindless_scrolling"),
    (r"\binstant\s+gratification\b", "instant_gratification"),
    (r"\bcheap\s+dopamine\b", "cheap_dopamine"),
    (r"\bover[\s\-]?stimulation\b", "overstimulation"),

    # Generation phrases
    (r"\bgen(?:eration)?\s+alpha\b", "genalpha"),
    (r"\bgen(?:eration)?\s+z(?:ee)?\b", "genz"),
    (r"\bgen(?:eration)?\s+x\b", "genx"),
    (r"\bipad\s+kids?\b", "ipadkids"),
    (r"\byounger\s+generation[s]?\b", "younger_generation"),
    (r"\bolder\s+generation[s]?\b", "older_generation"),
    (r"\bgeneration\s+gap\b", "generation_gap"),
    (r"\bkids\s+these\s+days\b", "kids_these_days"),

    # Language / communication phrases
    (r"\binternet\s+slang\b", "internet_slang"),
    (r"\bgen\s+z\s+slang\b", "genz_slang"),
    (r"\bgen\s+alpha\s+slang\b", "genalpha_slang"),
    (r"\bsocial\s+skills\b", "social_skills"),
    (r"\breal\s+life\b", "real_life"),
    (r"\bin\s+real\s+life\b", "real_life"),
    (r"\bcomment\s+section\b", "comment_section"),

    # Meme/slang phrases
    (r"\bno[\s\-]?cap\b", "nocap"),
    (r"\blow[\s\-]?key\b", "lowkey"),
    (r"\bhigh[\s\-]?key\b", "highkey"),
    (r"\bfanum\s+tax\b", "fanum_tax"),
    (r"\baura\s+points?\b", "aura_points"),
    (r"\bskibidi\s+toilet\b", "skibidi_toilet"),

    # Mental health / concern phrases
    (r"\bmental\s+health\b", "mental_health"),
    (r"\bbad\s+influence\b", "bad_influence"),
    (r"\bchild\s+development\b", "child_development"),
    (r"\bgrowing\s+up\b", "growing_up"),
]

def apply_replacements(text: str) -> str:
    for pattern, replacement in TEXT_REPLACEMENTS:
        text = re.sub(pattern, replacement, text)
    return text

df["network_text"] = df["network_text"].apply(apply_replacements)

# Remove hyphens that remain (they were protected during punctuation removal)
df["network_text"] = df["network_text"].apply(
    lambda t: re.sub(r"-", " ", t)
)
df["network_text"] = df["network_text"].apply(
    lambda t: re.sub(r"\s+", " ", t).strip()
)

print("Standardisation complete.")
print("Sample:", df["network_text"].iloc[3])


Standardisation complete.
Sample: having completely no context for it before this video i thought it was a solo version of 69


In [5]:
# Term -> theme dictionary
# Each key is a detected node/term.
# Each value is the broader concept/theme.

TERM_THEMES = {
    # Brainrot / overstimulation
    "brainrot": "brainrot",
    "braincells": "brainrot",
    "rot": "brainrot",
    "rotting": "brainrot",
    "doomscrolling": "brainrot",
    "overstimulation": "brainrot",
    "dopamine": "brainrot",
    "stimulation": "brainrot",
    "reward": "brainrot",

    # Addiction / attention / cognition
    "addicted": "attention_addiction",
    "addiction": "attention_addiction",
    "addictive": "attention_addiction",
    "attention": "attention_addiction",
    "focus": "attention_addiction",
    "concentrate": "attention_addiction",
    "hooked": "attention_addiction",

    # Platforms / online spaces
    "youtube": "platforms",
    "yt": "platforms",
    "tiktok": "platforms",
    "reels": "platforms",
    "shorts": "platforms",
    "internet": "platforms",
    "online": "platforms",
    "platform": "platforms",
    "platforms": "platforms",
    "app": "platforms",
    "apps": "platforms",
    "algorithm": "platforms",
    "algorithms": "platforms",
    "feed": "platforms",

    # Content / media consumption
    "content": "content_consumption",
    "consume": "content_consumption",
    "consuming": "content_consumption",
    "consumption": "content_consumption",
    "watch": "content_consumption",
    "watching": "content_consumption",
    "video": "content_consumption",
    "videos": "content_consumption",
    "channel": "content_consumption",
    "creator": "content_consumption",
    "binge": "content_consumption",
    "entertainment": "content_consumption",
    "recommendations": "content_consumption",
    "recommended": "content_consumption",
    "podcast": "content_consumption",
    "documentary": "content_consumption",
    "essay": "content_consumption",

    # Meme / slang culture
    "meme": "meme_slang",
    "memes": "meme_slang",
    "viral": "meme_slang",
    "trend": "meme_slang",
    "trends": "meme_slang",
    "slang": "meme_slang",
    "slangs": "meme_slang",
    "skibidi": "meme_slang",
    "toilet": "meme_slang",
    "sigma": "meme_slang",
    "alpha": "meme_slang",
    "rizz": "meme_slang",
    "gyatt": "meme_slang",
    "ohio": "meme_slang",
    "sus": "meme_slang",
    "cap": "meme_slang",
    "based": "meme_slang",
    "bro": "meme_slang",
    "bruh": "meme_slang",
    "fr": "meme_slang",
    "real": "meme_slang",
    "cooked": "meme_slang",
    "mid": "meme_slang",
    "npc": "meme_slang",
    "vibe": "meme_slang",
    "mewing": "meme_slang",
    "highkey": "meme_slang",
    "lowkey": "meme_slang",

    # Generations / age groups
    "genz": "generational_discussion",
    "boomer": "generational_discussion",
    "millennial": "generational_discussion",
    "millennials": "generational_discussion",
    "kid": "generational_discussion",
    "kids": "generational_discussion",
    "child": "generational_discussion",
    "children": "generational_discussion",
    "childhood": "generational_discussion",

    # Language / communication
    "language": "language_communication",
    "english": "language_communication",
    "word": "language_communication",
    "words": "language_communication",
    "meaning": "language_communication",
    "meanings": "language_communication",
    "vocabulary": "language_communication",
    "phrase": "language_communication",
    "phrases": "language_communication",
    "communicate": "language_communication",
    "communication": "language_communication",
    "conversation": "language_communication",
    "talk": "language_communication",
    "speaking": "language_communication",
    "saying": "language_communication",
    "understand": "language_communication",
    "understanding": "language_communication",
    "confused": "language_communication",
    "explain": "language_communication",
    "explained": "language_communication",
    "interaction": "language_communication",
    "comments": "language_communication",

    # Education / learning / literacy
    "education": "education_learning",
    "learn": "education_learning",
    "learning": "education_learning",
    "school": "education_learning",
    "student": "education_learning",
    "students": "education_learning",
    "teacher": "education_learning",
    "class": "education_learning",
    "study": "education_learning",
    "studying": "education_learning",
    "read": "education_learning",
    "reading": "education_learning",
    "write": "education_learning",
    "writing": "education_learning",
    "grammar": "education_learning",
    "book": "education_learning",
    "books": "education_learning",

    # Parenting / family / child development
    "parent": "parenting_childhood",
    "parents": "parenting_childhood",
    "parenting": "parenting_childhood",
    "mom": "parenting_childhood",
    "dad": "parenting_childhood",
    "mother": "parenting_childhood",
    "father": "parenting_childhood",
    "family": "parenting_childhood",
    "home": "parenting_childhood",
    "raise": "parenting_childhood",
    "raising": "parenting_childhood",
    "development": "parenting_childhood",

    # Devices / screen use
    "screen": "devices_screen_use",
    "screens": "devices_screen_use",
    "phone": "devices_screen_use",
    "phones": "devices_screen_use",
    "ipad": "devices_screen_use",
    "tablet": "devices_screen_use",
    "device": "devices_screen_use",
    "devices": "devices_screen_use",
    "mobile": "devices_screen_use",
    "smartphone": "devices_screen_use",
    "scroll": "devices_screen_use",
    "scrolling": "devices_screen_use",
    "swiping": "devices_screen_use",

    # Social concern / impact
    "problem": "social_concern",
    "issue": "social_concern",
    "concern": "social_concern",
    "worried": "social_concern",
    "worry": "social_concern",
    "dangerous": "social_concern",
    "harmful": "social_concern",
    "affecting": "social_concern",
    "impact": "social_concern",
    "effect": "social_concern",
    "effects": "social_concern",
    "anxiety": "social_concern",
    "depression": "social_concern",

    # Negative judgement / criticism
    "bad": "negative_judgement",
    "terrible": "negative_judgement",
    "awful": "negative_judgement",
    "annoying": "negative_judgement",
    "cringe": "negative_judgement",
    "hate": "negative_judgement",
    "sad": "negative_judgement",
    "scary": "negative_judgement",
    "trash": "negative_judgement",
    "garbage": "negative_judgement",
    "ruined": "negative_judgement",
    "destroyed": "negative_judgement",
    "worse": "negative_judgement",
    "worst": "negative_judgement",
    "dumb": "negative_judgement",
    "stupid": "negative_judgement",
    "nonsense": "negative_judgement",
    "weird": "negative_judgement",

    # Positive judgement / appreciation
    "good": "positive_judgement",
    "great": "positive_judgement",
    "amazing": "positive_judgement",
    "interesting": "positive_judgement",
    "funny": "positive_judgement",
    "love": "positive_judgement",
    "enjoy": "positive_judgement",
    "helpful": "positive_judgement",
    "useful": "positive_judgement",
    "smart": "positive_judgement",
    "thanks": "positive_judgement",
    "silly": "positive_judgement",

    # Community / audience
    "audience": "community_audience",
    "community": "community_audience",
    "communities": "community_audience",
    "friend": "community_audience",
    "friends": "community_audience",
    "irl": "community_audience",
    "outside": "community_audience",

    # Time / change over time
    "change": "change_over_time",
    "changed": "change_over_time",
    "changing": "change_over_time",
    "current": "change_over_time",
    "modern": "change_over_time",
    "nowadays": "change_over_time",
    "today": "change_over_time",
    "future": "change_over_time",
    "nostalgia": "change_over_time",

    # Identity / personality
    "personality": "identity_behaviour",
    "random": "identity_behaviour",
}

# Create vocabulary from all terms in all lists (flatten the lists)
VOCABULARY = set(TERM_THEMES.keys())

print(f"Vocabulary: {len(VOCABULARY)} terms across {len(set(TERM_THEMES.values()))} themes")
print(f"Terms: {sorted(VOCABULARY)}")

Vocabulary: 209 terms across 16 themes
Terms: ['addicted', 'addiction', 'addictive', 'affecting', 'algorithm', 'algorithms', 'alpha', 'amazing', 'annoying', 'anxiety', 'app', 'apps', 'attention', 'audience', 'awful', 'bad', 'based', 'binge', 'book', 'books', 'boomer', 'braincells', 'brainrot', 'bro', 'bruh', 'cap', 'change', 'changed', 'changing', 'channel', 'child', 'childhood', 'children', 'class', 'comments', 'communicate', 'communication', 'communities', 'community', 'concentrate', 'concern', 'confused', 'consume', 'consuming', 'consumption', 'content', 'conversation', 'cooked', 'creator', 'cringe', 'current', 'dad', 'dangerous', 'depression', 'destroyed', 'development', 'device', 'devices', 'documentary', 'doomscrolling', 'dopamine', 'dumb', 'education', 'effect', 'effects', 'english', 'enjoy', 'entertainment', 'essay', 'explain', 'explained', 'family', 'father', 'feed', 'focus', 'fr', 'friend', 'friends', 'funny', 'future', 'garbage', 'genz', 'good', 'grammar', 'great', 'gyatt', 

In [6]:
# Detect relevant terms per comment

def detect_terms(text: str) -> list:
    # Split on whitespace to get exact tokens; no partial matches
    tokens = set(text.split())
    return sorted(tokens & VOCABULARY)

df["detected_terms"] = df["network_text"].apply(detect_terms)
df["term_count"] = df["detected_terms"].apply(len)

print(f"Comments with 0 terms:  {(df['term_count'] == 0).sum():,}")
print(f"Comments with 1 term:   {(df['term_count'] == 1).sum():,}")
print(f"Comments with >=2 terms: {(df['term_count'] >= 2).sum():,}")

# Term frequency overview
all_detected = [t for terms in df["detected_terms"] for t in terms]
term_freq = Counter(all_detected)
print(f"\nTop 20 detected terms:")
for term, count in term_freq.most_common(20):
    print(f"  {term:<22} {count:>5}  ({TERM_THEMES.get(term, 'unknown')})")


Comments with 0 terms:  1,539
Comments with 1 term:   1,671
Comments with >=2 terms: 3,031

Top 20 detected terms:
  video                    945  (content_consumption)
  brainrot                 498  (brainrot)
  kids                     336  (generational_discussion)
  good                     329  (positive_judgement)
  phone                    321  (devices_screen_use)
  watching                 304  (content_consumption)
  content                  301  (content_consumption)
  love                     276  (positive_judgement)
  watch                    271  (content_consumption)
  videos                   252  (content_consumption)
  great                    224  (positive_judgement)
  genz                     221  (generational_discussion)
  youtube                  220  (platforms)
  thanks                   199  (positive_judgement)
  slang                    197  (meme_slang)
  internet                 191  (platforms)
  parents                  186  (parenting_childhood)
  sc

In [7]:
# Keep Comments with at Least 2 Detected Terms

df_network = df[df["term_count"] >= 2].reset_index(drop=True)

print(f"Before filter: {len(df):,} comments")
print(f"After filter (>= 2 terms): {len(df_network):,} comments "
      f"({len(df_network)/len(df)*100:.1f}% of corpus)")
df_network[["network_text", "detected_terms", "term_count"]].head(5)


Before filter: 6,241 comments
After filter (>= 2 terms): 3,031 comments (48.6% of corpus)


,network_text,detected_terms,term_count
0,nothing makes me feel older then watching a 15...,"[kids, slang, video, watching]",4
1,the next time my kid says 6 7 i m going to pun...,"[kid, video, watch]",3
2,nobody tell youtube that they re hosting insig...,"[content, youtube]",2
3,i ve heard that dissecting a joke is like diss...,"[learn, thanks]",2
4,being philly born and raised this video is gre...,"[great, kids, video]",3


In [8]:
# Generate Term Pairs and Count Edge Weights

edge_counter = Counter()

for terms in df_network["detected_terms"]:
    # combinations gives each unordered pair exactly once per comment
    for pair in combinations(sorted(terms), 2):
        edge_counter[pair] += 1

edges_df = pd.DataFrame(
    [(src, tgt, w) for (src, tgt), w in edge_counter.items()],
    columns=["source", "target", "weight"]
).sort_values("weight", ascending=False).reset_index(drop=True)

print(f"Total unique term pairs (raw edges): {len(edges_df):,}")
print(f"\nWeight distribution:")
print(edges_df["weight"].describe().round(2))
print(f"\nTop 20 edges:")
edges_df.head(20)


Total unique term pairs (raw edges): 8,721

Weight distribution:
count    8721.00
mean        3.17
std         5.04
min         1.00
25%         1.00
50%         2.00
75%         3.00
max        96.00
Name: weight, dtype: float64

Top 20 edges:


,source,target,weight
0,great,video,96
1,video,watching,88
2,video,watch,81
3,brainrot,video,77
4,thanks,video,73
5,good,video,72
6,kids,parents,70
7,content,video,62
8,phone,video,62
9,love,video,61


In [9]:
# Filter Weak Edges and Remove Isolated Nodes

# Adjust threshold based on the weight distribution shown in Cell 8.
# weight >= 3 keeps more edges (denser graph)
# weight >= 5 keeps fewer edges (cleaner, stronger connections only)
WEIGHT_THRESHOLD = 3

edges_filtered = edges_df[edges_df["weight"] >= WEIGHT_THRESHOLD].copy()
print(f"Edges after weight >= {WEIGHT_THRESHOLD}: {len(edges_filtered):,} "
      f"(removed {len(edges_df) - len(edges_filtered):,})")

# Nodes that appear in at least one retained edge (no isolated nodes)
active_nodes = set(edges_filtered["source"]) | set(edges_filtered["target"])
print(f"\nActive nodes (non-isolated): {len(active_nodes):,}")
print(f"Removed isolated nodes: "
      f"{len(VOCABULARY) - len(active_nodes)} term(s) never co-occur above threshold")
print(f"Isolated terms: {sorted(VOCABULARY - active_nodes)}")


Edges after weight >= 3: 2,887 (removed 5,834)

Active nodes (non-isolated): 209
Removed isolated nodes: 0 term(s) never co-occur above threshold
Isolated terms: []


In [10]:
# Save edge list and node list

# --- Edge list ---
edges_filtered.to_csv("data_network_analysis.csv", index=False, encoding="utf-8")
print(f"Saved edge list: data_network_analysis.csv  ({len(edges_filtered):,} edges)")

# --- Node list (with theme labels, will be enriched with metrics in Cell 12) ---
nodes_df = pd.DataFrame({
    "node": sorted(active_nodes),
    "theme": [TERM_THEMES.get(n, "unknown") for n in sorted(active_nodes)],
})
nodes_df.to_csv("data_network_analysis_nodes.csv", index=False, encoding="utf-8")
print(f"Saved node list: data_network_analysis_nodes.csv  ({len(nodes_df):,} nodes)")

edges_filtered.head(10)


Saved edge list: data_network_analysis.csv  (2,887 edges)
Saved node list: data_network_analysis_nodes.csv  (209 nodes)


,source,target,weight
0,great,video,96
1,video,watching,88
2,video,watch,81
3,brainrot,video,77
4,thanks,video,73
5,good,video,72
6,kids,parents,70
7,content,video,62
8,phone,video,62
9,love,video,61


In [ ]:
# Network analysis

In [ ]:
# Build NetworkX Graph

G = nx.Graph()

# Add nodes with theme attribute
for _, row in nodes_df.iterrows():
    G.add_node(row["node"], theme=row["theme"])

# Add weighted edges
for _, row in edges_filtered.iterrows():
    G.add_edge(row["source"], row["target"], weight=row["weight"])

print(f"Graph summary:")
print(f"  Nodes:   {G.number_of_nodes()}")
print(f"  Edges:   {G.number_of_edges()}")
print(f"  Density: {nx.density(G):.4f}")
print(f"  Connected: {nx.is_connected(G)}")
if not nx.is_connected(G):
    components = list(nx.connected_components(G))
    print(f"  Connected components: {len(components)}")
    sizes = sorted([len(c) for c in components], reverse=True)
    print(f"  Component sizes: {sizes}")


In [ ]:
# Calculate Centrality, Communities and Density

# --- Weighted degree (strength) ---
weighted_degree = dict(G.degree(weight="weight"))

# --- Degree centrality (unweighted, normalised) ---
degree_centrality = nx.degree_centrality(G)

# --- Betweenness centrality ---
# NetworkX treats 'weight' as distance — use inverse weight so
# higher co-occurrence = shorter distance = more central path
for u, v, d in G.edges(data=True):
    G[u][v]["inv_weight"] = 1.0 / d["weight"]

betweenness = nx.betweenness_centrality(G, weight="inv_weight", normalized=True)

# --- Closeness centrality ---
closeness = nx.closeness_centrality(G)

# --- PageRank (weighted) ---
pagerank = nx.pagerank(G, weight="weight", alpha=0.85)

# --- Community detection (greedy modularity, built into NetworkX) ---
communities_raw = list(greedy_modularity_communities(G, weight="weight"))
community_map = {}
for comm_id, members in enumerate(communities_raw):
    for node in members:
        community_map[node] = comm_id

modularity = nx.algorithms.community.quality.modularity(
    G, communities_raw, weight="weight"
)

print(f"Communities detected: {len(communities_raw)}")
print(f"Modularity score:     {modularity:.4f}  (>0.3 indicates meaningful structure)")
print(f"\nCommunity composition:")
for i, comm in enumerate(communities_raw):
    members = sorted(comm)
    print(f"  Community {i}: {members}")

# --- Compile node metrics ---
metrics_df = pd.DataFrame({
    "node":                 list(G.nodes()),
    "theme":                [G.nodes[n].get("theme", "") for n in G.nodes()],
    "community":            [community_map[n] for n in G.nodes()],
    "degree":               [G.degree(n) for n in G.nodes()],
    "weighted_degree":      [weighted_degree[n] for n in G.nodes()],
    "degree_centrality":    [round(degree_centrality[n], 4) for n in G.nodes()],
    "betweenness":          [round(betweenness[n], 4) for n in G.nodes()],
    "closeness":            [round(closeness[n], 4) for n in G.nodes()],
    "pagerank":             [round(pagerank[n], 4) for n in G.nodes()],
}).sort_values("weighted_degree", ascending=False).reset_index(drop=True)

# Overwrite node CSV with full metrics
metrics_df.to_csv("data_network_analysis_nodes.csv", index=False, encoding="utf-8")
print(f"\nNode metrics saved to data_network_analysis_nodes.csv")

print(f"\nTop 15 nodes by weighted degree:")
print(metrics_df[["node","theme","community","degree","weighted_degree",
                   "betweenness","pagerank"]].head(15).to_string(index=False))

print(f"\n--- Graph Statistics ---")
print(f"Nodes:              {G.number_of_nodes()}")
print(f"Edges:              {G.number_of_edges()}")
print(f"Density:            {nx.density(G):.4f}")
print(f"Avg weighted degree:{sum(weighted_degree.values())/len(weighted_degree):.2f}")
print(f"Communities:        {len(communities_raw)}")
print(f"Modularity:         {modularity:.4f}")
